# City Library Management System

This notebook uses CSV files as the data source for books, members, and borrow records.

In [2]:
import csv
import os
from pathlib import Path

BASE_DIR = Path.cwd()
BOOKS_FILE = BASE_DIR / 'books.csv'
MEMBERS_FILE = BASE_DIR / 'members.csv'
BORROW_LOG_FILE = BASE_DIR / 'borrow_log.csv'

def load_csv_rows(file_path):
    if not file_path.exists():
        return []
    with open(file_path, newline='', encoding='utf-8') as file:
        return list(csv.DictReader(file))

def save_csv_rows(file_path, rows, fieldnames):
    with open(file_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def initialize_data_files():
    if not BOOKS_FILE.exists():
        sample_books = [
            {'Book ID': 'B001', 'Title': 'The Hobbit', 'Author': 'J.R.R. Tolkien', 'Genre': 'Fantasy', 'Availability': 'Available'},
            {'Book ID': 'B002', 'Title': '1984', 'Author': 'George Orwell', 'Genre': 'Dystopian', 'Availability': 'Available'},
            {'Book ID': 'B003', 'Title': 'Atomic Habits', 'Author': 'James Clear', 'Genre': 'Self-Help', 'Availability': 'Available'},
        ]
        save_csv_rows(BOOKS_FILE, sample_books, ['Book ID', 'Title', 'Author', 'Genre', 'Availability'])

    if not MEMBERS_FILE.exists():
        sample_members = [
            {'Member ID': 'M001', 'Name': 'Ava', 'Age': '25', 'Contact Info': 'ava@email.com'},
            {'Member ID': 'M002', 'Name': 'Ben', 'Age': '31', 'Contact Info': 'ben@email.com'},
        ]
        save_csv_rows(MEMBERS_FILE, sample_members, ['Member ID', 'Name', 'Age', 'Contact Info'])

    if not BORROW_LOG_FILE.exists():
        save_csv_rows(BORROW_LOG_FILE, [], ['Transaction ID', 'Member ID', 'Book ID', 'Action', 'Date'])

initialize_data_files()

def load_books():
    return load_csv_rows(BOOKS_FILE)

def load_members():
    return load_csv_rows(MEMBERS_FILE)

def load_borrow_log():
    return load_csv_rows(BORROW_LOG_FILE)

def save_books(books):
    save_csv_rows(BOOKS_FILE, books, ['Book ID', 'Title', 'Author', 'Genre', 'Availability'])

def save_members(members):
    save_csv_rows(MEMBERS_FILE, members, ['Member ID', 'Name', 'Age', 'Contact Info'])

def save_borrow_log(log):
    save_csv_rows(BORROW_LOG_FILE, log, ['Transaction ID', 'Member ID', 'Book ID', 'Action', 'Date'])

def add_book(book_id, title, author, genre, availability='Available'):
    books = load_books()
    if any(book['Book ID'] == book_id for book in books):
        raise ValueError('Book ID already exists')
    books.append({'Book ID': book_id, 'Title': title, 'Author': author, 'Genre': genre, 'Availability': availability})
    save_books(books)
    return f'Added book: {title}'

def add_member(member_id, name, age, contact_info):
    members = load_members()
    if any(member['Member ID'] == member_id for member in members):
        raise ValueError('Member ID already exists')
    members.append({'Member ID': member_id, 'Name': name, 'Age': age, 'Contact Info': contact_info})
    save_members(members)
    return f'Added member: {name}'

def issue_book(member_id, book_id, date='2026-08-02'):
    members = load_members()
    books = load_books()
    log = load_borrow_log()

    member = next((m for m in members if m['Member ID'] == member_id), None)
    book = next((b for b in books if b['Book ID'] == book_id), None)

    if not member:
        raise ValueError('Member not found')
    if not book:
        raise ValueError('Book not found')
    if book['Availability'] != 'Available':
        raise ValueError('Book is already issued')

    book['Availability'] = 'Issued'
    save_books(books)
    log.append({'Transaction ID': f'T{len(log) + 1:03d}', 'Member ID': member_id, 'Book ID': book_id, 'Action': 'Borrow', 'Date': date})
    save_borrow_log(log)
    return f"{book['Title']} issued to {member['Name']}"

def return_book(member_id, book_id, date='2026-08-02'):
    books = load_books()
    log = load_borrow_log()

    book = next((b for b in books if b['Book ID'] == book_id), None)
    if not book:
        raise ValueError('Book not found')
    if book['Availability'] != 'Issued':
        raise ValueError('Book is not currently issued')

    book['Availability'] = 'Available'
    save_books(books)
    log.append({'Transaction ID': f'T{len(log) + 1:03d}', 'Member ID': member_id, 'Book ID': book_id, 'Action': 'Return', 'Date': date})
    save_borrow_log(log)
    return f"{book['Title']} returned successfully"

def search_books(query):
    query = query.lower()
    return [book for book in load_books() if query in book['Title'].lower() or query in book['Author'].lower()]

def show_available_books_by_genre(genre):
    return [book for book in load_books() if book['Genre'].lower() == genre.lower() and book['Availability'] == 'Available']

def list_members_who_borrowed():
    members = load_members()
    log = load_borrow_log()
    borrowed_ids = {entry['Member ID'] for entry in log if entry['Action'] == 'Borrow'}
    return [member for member in members if member['Member ID'] in borrowed_ids]

def get_most_popular_genre():
    books = load_books()
    log = load_borrow_log()
    genre_count = {}
    for entry in log:
        if entry['Action'] != 'Borrow':
            continue
        book = next((b for b in books if b['Book ID'] == entry['Book ID']), None)
        if book:
            genre_count[book['Genre']] = genre_count.get(book['Genre'], 0) + 1
    if not genre_count:
        return 'No borrow records yet'
    return max(genre_count.items(), key=lambda item: item[1])[0]

def generate_library_summary():
    books = load_books()
    members = load_members()
    available_count = sum(1 for book in books if book['Availability'] == 'Available')
    issued_count = sum(1 for book in books if book['Availability'] == 'Issued')
    return {
        'Total Books': len(books),
        'Total Members': len(members),
        'Available Books': available_count,
        'Issued Books': issued_count,
        'Popular Genre': get_most_popular_genre()
    }

In [ ]:
# Example usage
print(add_book('B004', 'The Alchemist', 'Paulo Coelho', 'Adventure'))
print(add_member('M003', 'Chloe', '27', 'chloe@email.com'))
print(issue_book('M001', 'B001'))
print(return_book('M001', 'B001'))
print(show_available_books_by_genre('Fantasy'))
print(list_members_who_borrowed())
print(search_books('tolkien'))
print(get_most_popular_genre())
print(generate_library_summary())

ValueError: Book ID already exists